# 07 — Análisis ACWR

**TFM RunnAing** | Fase 6.2

**Acute:Chronic Workload Ratio** sobre TRIMP predicho por el modelo ganador.

```
ACWR = media_TRIMP(ultimos 7 dias) / media_TRIMP(ultimos 28 dias)
```

**Zonas (Gabbett, 2016):**
- 0.8 <= ACWR <= 1.3  -> Sweet spot (carga óptima)
- ACWR > 1.5          -> Zona de riesgo elevado (sobreentrenamiento)
- Resto               -> Zona de precaución

Solo para usuarios con span >= 28 días (cohorte apta para ACWR canónico).

**Outputs**: `reports/figures/acwr_*.png` + `reports/evaluation/acwr_zonas.csv`

In [ ]:
import sys
sys.path.insert(0, '..')

import pickle
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from pathlib import Path

from src.splits import group_train_val_test_split

np.random.seed(42)
MODELS_DIR = Path('../models')
FIG_DIR = Path('../reports/figures')
FIG_DIR.mkdir(parents=True, exist_ok=True)
EVAL_DIR = Path('../reports/evaluation')
EVAL_DIR.mkdir(parents=True, exist_ok=True)

# Zonas ACWR según especificación
SWEET_LOW, SWEET_HIGH = 0.8, 1.3
RISK_HIGH = 1.5

def assign_zone(acwr):
    """Clasifica un valor ACWR en su zona de riesgo."""
    if pd.isna(acwr):
        return 'Sin datos'
    if SWEET_LOW <= acwr <= SWEET_HIGH:
        return 'Sweet spot'
    if acwr > RISK_HIGH:
        return 'Riesgo elevado'
    return 'Precaución'

print('Zonas:')
print(f'  Sweet spot    : {SWEET_LOW} <= ACWR <= {SWEET_HIGH}')
print(f'  Precaucion    : resto')
print(f'  Riesgo elevado: ACWR > {RISK_HIGH}')

## 1. Cargar modelo y generar predicciones sobre dataset completo

In [ ]:
with open(MODELS_DIR / 'split_meta.json') as f:
    meta = json.load(f)
best_name = meta['best_model_name']
FEATURE_COLS = meta['feature_cols']

with open(MODELS_DIR / 'best_model.pkl', 'rb') as f:
    model = pickle.load(f)

df = pd.read_parquet('../data/processed/sessions_features.parquet')

# Predecir TRIMP sobre TODO el dataset (train+val+test)
# ACWR requiere la serie temporal completa por usuario
X_all = df[FEATURE_COLS]
df['trimp_pred'] = model.predict(X_all)

print(f'Modelo: {best_name}')
print(f'Sesiones totales con prediccion: {len(df):,}')
print(f'TRIMP predicho — media={df.trimp_pred.mean():.2f}, std={df.trimp_pred.std():.2f}')

## 2. Cohorte apta para ACWR (span >= 28 días)

Solo usuarios con historial temporal suficiente para calcular la ventana crónica de 28 días.

In [ ]:
ACWR_THRESHOLD = 28

span_stats = df.groupby('userId')['date'].agg(
    primera='min', ultima='max', n_sesiones='count'
).reset_index()
span_stats['span_dias'] = (span_stats['ultima'] - span_stats['primera']).dt.days

eligible_users = span_stats[span_stats['span_dias'] >= ACWR_THRESHOLD]['userId']
n_eligible = len(eligible_users)
n_total = span_stats['userId'].nunique()
n_excluded = n_total - n_eligible

print(f'Usuarios totales   : {n_total:,}')
print(f'Aptos (span>=28d)  : {n_eligible:,}  ({100*n_eligible/n_total:.1f}%)')
print(f'Excluidos          : {n_excluded:,}  ({100*n_excluded/n_total:.1f}%)')

df_cohort = df[df['userId'].isin(eligible_users)].copy()
print(f'Sesiones en cohorte: {len(df_cohort):,}')

## 3. Cálculo del ACWR por usuario

In [ ]:
def compute_acwr_for_user(user_df: pd.DataFrame) -> pd.DataFrame:
    """Calcula ACWR diario para un usuario: rolling 7d / rolling 28d sobre TRIMP predicho."""
    # Agregar TRIMP predicho por dia
    daily = (
        user_df.groupby('date')['trimp_pred']
        .sum()
        .reset_index()
        .sort_values('date')
    )
    daily = daily.set_index('date')
    # Rellenar dias sin sesion con 0
    full_range = pd.date_range(daily.index.min(), daily.index.max(), freq='D')
    daily = daily.reindex(full_range, fill_value=0.0)

    # Rolling con min_periods para no producir NaN en los primeros dias
    acute  = daily['trimp_pred'].rolling(7,  min_periods=1).mean()
    chronic = daily['trimp_pred'].rolling(28, min_periods=7).mean()

    acwr = acute / chronic.replace(0, np.nan)
    return pd.DataFrame({'date': daily.index, 'acute': acute.values,
                         'chronic': chronic.values, 'acwr': acwr.values})

acwr_records = []
for user_id, user_df in df_cohort.groupby('userId'):
    user_acwr = compute_acwr_for_user(user_df)
    user_acwr['userId'] = user_id
    acwr_records.append(user_acwr)

acwr_df = pd.concat(acwr_records, ignore_index=True)
acwr_df['zona'] = acwr_df['acwr'].apply(assign_zone)

print(f'Filas ACWR diario: {len(acwr_df):,}')
print(f'ACWR valido: {acwr_df.acwr.notna().sum():,}')
print('\nDistribucion de zonas:')
print(acwr_df['zona'].value_counts().to_string())

## 4. ACWR final por usuario (último valor disponible)

Para clasificar el estado actual de cada usuario en la cohorte.

In [ ]:
# ACWR final = ultimo valor valido por usuario
last_acwr = (
    acwr_df[acwr_df['acwr'].notna()]
    .sort_values('date')
    .groupby('userId')
    .last()
    .reset_index()
    [['userId', 'date', 'acwr', 'zona']]
)

print(f'Usuarios con ACWR calculado: {len(last_acwr):,}')
print('\nDistribucion de zonas (ultimo ACWR por usuario):')
zone_counts = last_acwr['zona'].value_counts()
for zona, n in zone_counts.items():
    pct = 100 * n / len(last_acwr)
    symbol = 'OK' if zona == 'Sweet spot' else ('!!' if zona == 'Riesgo elevado' else 'Precaucion')
    print(f'  [{symbol}] {zona:<18}: {n:>5,} usuarios ({pct:.1f}%)')

## 5. Tabla de usuarios en zona de riesgo elevado

In [ ]:
risk_users = last_acwr[last_acwr['zona'] == 'Riesgo elevado'].copy()
risk_users = risk_users.sort_values('acwr', ascending=False)

print(f'Usuarios en riesgo elevado (ACWR > {RISK_HIGH}): {len(risk_users):,}')
if len(risk_users) > 0:
    print(risk_users.head(20).to_string(index=False))

# Guardar tablas
last_acwr.to_csv(EVAL_DIR / 'acwr_zonas.csv', index=False)
risk_users.to_csv(EVAL_DIR / 'acwr_usuarios_riesgo.csv', index=False)
print('\nGuardado: reports/evaluation/acwr_zonas.csv')
print('Guardado: reports/evaluation/acwr_usuarios_riesgo.csv')

## 6. Visualizaciones

In [ ]:
# Distribucion del ultimo ACWR por usuario
fig, ax = plt.subplots(figsize=(9, 5))

acwr_vals = last_acwr['acwr'].dropna()
ax.hist(acwr_vals, bins=60, color='steelblue', edgecolor='white', alpha=0.85)

# Zonas
ax.axvspan(SWEET_LOW, SWEET_HIGH, alpha=0.15, color='green', label='Sweet spot (0.8-1.3)')
ax.axvspan(RISK_HIGH, acwr_vals.max() * 1.05, alpha=0.15, color='red',
           label=f'Riesgo elevado (>{RISK_HIGH})')
ax.axvline(SWEET_LOW, color='green', linestyle='--', linewidth=1.2)
ax.axvline(SWEET_HIGH, color='orange', linestyle='--', linewidth=1.2)
ax.axvline(RISK_HIGH, color='red', linestyle='--', linewidth=1.5)

ax.set_xlabel('ACWR (ultimo valor por usuario)', fontsize=12)
ax.set_ylabel('Numero de usuarios', fontsize=12)
ax.set_title('Distribucion del ACWR por usuario — cohorte apta (span >= 28 dias)',
             fontsize=12)
ax.legend(fontsize=10)
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig(FIG_DIR / 'acwr_distribucion.png', dpi=150, bbox_inches='tight')
plt.show()
print('Guardado: reports/figures/acwr_distribucion.png')

In [ ]:
# Grafico de zonas
zone_order = ['Sweet spot', 'Precaucion', 'Riesgo elevado']
zone_counts_ordered = [zone_counts.get(z, 0) for z in zone_order]
zone_colors = ['#2ecc71', '#f39c12', '#e74c3c']

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Pie chart
wedges, texts, autotexts = axes[0].pie(
    zone_counts_ordered, labels=zone_order, colors=zone_colors,
    autopct='%1.1f%%', startangle=90,
    textprops={'fontsize': 11}
)
axes[0].set_title('Distribucion de zonas ACWR', fontsize=12)

# Bar chart
bars = axes[1].bar(zone_order, zone_counts_ordered,
                   color=zone_colors, edgecolor='white')
for bar, n in zip(bars, zone_counts_ordered):
    pct = 100 * n / sum(zone_counts_ordered)
    axes[1].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1,
                 f'{n}\n({pct:.1f}%)', ha='center', fontsize=10)
axes[1].set_title('Usuarios por zona', fontsize=12)
axes[1].set_ylabel('Numero de usuarios')
axes[1].grid(axis='y', alpha=0.3)

plt.suptitle('Clasificacion de riesgo ACWR — cohorte apta', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(FIG_DIR / 'acwr_zonas.png', dpi=150, bbox_inches='tight')
plt.show()
print('Guardado: reports/figures/acwr_zonas.png')

In [ ]:
# Evolucion temporal del ACWR para 3 usuarios de ejemplo (uno por zona)
sample_users = {}
for zona in ['Sweet spot', 'Precaucion', 'Riesgo elevado']:
    subset = last_acwr[last_acwr['zona'] == zona]
    if len(subset) > 0:
        sample_users[zona] = subset.iloc[0]['userId']

fig, axes = plt.subplots(len(sample_users), 1, figsize=(13, 4*len(sample_users)))
if len(sample_users) == 1:
    axes = [axes]

zone_color_map = {'Sweet spot': '#2ecc71', 'Precaucion': '#f39c12', 'Riesgo elevado': '#e74c3c'}

for ax, (zona, uid) in zip(axes, sample_users.items()):
    user_ts = acwr_df[acwr_df['userId'] == uid].sort_values('date')
    ax.plot(user_ts['date'], user_ts['acwr'], color='steelblue', linewidth=1.5)
    ax.axhspan(SWEET_LOW, SWEET_HIGH, alpha=0.15, color='green')
    ax.axhspan(RISK_HIGH, max(user_ts['acwr'].max(), RISK_HIGH + 0.1), alpha=0.1, color='red')
    ax.axhline(SWEET_LOW, color='green', linestyle=':', linewidth=1)
    ax.axhline(SWEET_HIGH, color='orange', linestyle=':', linewidth=1)
    ax.axhline(RISK_HIGH, color='red', linestyle='--', linewidth=1.2)
    ax.set_title(f'Usuario ejemplo — zona: {zona}', fontsize=11)
    ax.set_ylabel('ACWR')
    ax.grid(alpha=0.3)

plt.suptitle('Evolucion temporal del ACWR por usuario', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig(FIG_DIR / 'acwr_timeline_ejemplos.png', dpi=150, bbox_inches='tight')
plt.show()
print('Guardado: reports/figures/acwr_timeline_ejemplos.png')

In [ ]:
print('=== RESUMEN FASE 6.2 ===')
print(f'Cohorte apta (span >= 28d)     : {len(last_acwr):,} usuarios')
print(f'Sweet spot  (0.8<=ACWR<=1.3)  : {zone_counts.get("Sweet spot", 0):>6,}')
print(f'Precaucion                     : {zone_counts.get("Precaucion", 0):>6,}')
print(f'Riesgo elevado (ACWR>1.5)      : {zone_counts.get("Riesgo elevado", 0):>6,}')
print(f'\nOutputs:')
print(f'  reports/evaluation/acwr_zonas.csv')
print(f'  reports/evaluation/acwr_usuarios_riesgo.csv')
print(f'  reports/figures/acwr_distribucion.png')
print(f'  reports/figures/acwr_zonas.png')
print(f'  reports/figures/acwr_timeline_ejemplos.png')